# File Handling - CSV Files

CSV (comma-separated values) stores a table as text. Each line is a row and each comma separates a column.

| Tool | Purpose |
|---|---|
| `csv.reader(file)` | Reads each row as a **list** |
| `csv.writer(file)` | Writes rows from lists |
| `writer.writerow(row)` | Write one row |
| `writer.writerows(rows)` | Write many rows |
| `csv.DictReader(file)` | Reads each row as a **dict** using the header |
| `csv.DictWriter(file, fieldnames)` | Writes rows from dicts |
| `writer.writeheader()` | Write the header row (`DictWriter`) |
| `delimiter=` | Column separator (`","`, `";"`, `"\t"`) |
| `newline=""` | Required when opening the file |

```python
import csv
```

---

## Always Open With `newline=""`

```python
with open("data.csv", "w", newline="", encoding="utf-8") as file:
    ...
```

Without `newline=""`, extra blank lines can appear on Windows, and line breaks inside quoted fields can break.

---

## Writing

```python
writer = csv.writer(file)
writer.writerow(["name", "age"])
writer.writerows([["Ann", 30], ["Bob", 25]])
```

The module adds quotes automatically when a value contains a comma, a quote or a line break.

---

## Reading

```python
reader = csv.reader(file)
header = next(reader)
for row in reader:
    ...
```

### Important

* **Every value is a string**, even numbers. Convert with `int()` or `float()`.
* An empty line gives an empty list.

---

## Dictionaries

```python
for row in csv.DictReader(file):
    print(row["name"], row["age"])

writer = csv.DictWriter(file, fieldnames=["name", "age"])
writer.writeheader()
writer.writerow({"name": "Ann", "age": 30})
```

`DictReader` uses the first row as keys. Rows are easier to read than positions.

---

## Other Separators

```python
csv.reader(file, delimiter=";")     # semicolon files
csv.reader(file, delimiter="\t")    # tab-separated files (TSV)
```

---

## Notes

* Excel may need `encoding="utf-8-sig"` to show non-English text correctly.
* For large tables and analysis, use `pandas` (`pd.read_csv`).

## Source

https://docs.python.org/3/library/csv.html

In [ ]:
import csv
import io
import tempfile
from pathlib import Path

rows = [["name", "age", "city"], ["Ann", 30, "Cairo"], ["Bob", 25, "Alexandria, Egypt"]]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "people.csv"

    # Writing: values containing commas are quoted automatically
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(rows[0])
        writer.writerows(rows[1:])
    print(path.read_text(encoding="utf-8"))

    # Reading with csv.reader: every value is a string
    with open(path, newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        header = next(reader)
        data = list(reader)
    print(header, data)
    print(sum(int(row[1]) for row in data))            # convert before calculating

    # DictReader: rows as dictionaries
    with open(path, newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            print(row["name"], "lives in", row["city"])

    # DictWriter
    out = Path(tmp) / "scores.csv"
    with open(out, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["name", "score"])
        writer.writeheader()
        writer.writerow({"name": "Ann", "score": 9.5})
        writer.writerow({"name": "Bob", "score": 7})
    print(out.read_text(encoding="utf-8").splitlines())

# Other separators (an in-memory file works the same way)
semicolon = io.StringIO("a;b\n1;2\n")
print(list(csv.reader(semicolon, delimiter=";")))

tabs = io.StringIO("x\ty\n5\t6\n")
print(list(csv.DictReader(tabs, delimiter="\t")))